In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt


In [ ]:

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:


# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors


# Convert X to float32 tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)

# Convert y to float32 tensors (regression)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

# If images are in NHWC format (N, H, W, C), convert to NCHW (N, C, H, W)
if X_train_t.ndim == 4 and X_train_t.shape[-1] in [1, 3]:
    X_train_t = X_train_t.permute(0, 3, 1, 2)  # NHWC -> NCHW
    X_test_t  = X_test_t.permute(0, 3, 1, 2)

# Ensure y is shape (N, 1) for regression
if y_train_t.ndim == 1:
    y_train_t = y_train_t.unsqueeze(1)
if y_test_t.ndim == 1:
    y_test_t = y_test_t.unsqueeze(1)

print("X_train_t shape:", X_train_t.shape)
print("y_train_t shape:", y_train_t.shape)



In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset  = TensorDataset(X_test_t,  y_test_t)




In [ ]:
# 3. Create DataLoaders (batch size = 32)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)


In [ ]:
# 4. Print shape of one batch
images_batch, labels_batch = next(iter(train_loader))
print("One batch images shape:", images_batch.shape)  # (32, C, H, W)
print("One batch labels shape:", labels_batch.shape)  # (32, 1)


In [ ]:
#5) Display a few images using matplotlib

# Display first 6 images from the batch
plt.figure(figsize=(12, 6))

num_show = 6
for i in range(num_show):
    img = images_batch[i]

    # Convert from (C, H, W) to (H, W, C) for plotting
    img_np = img.permute(1, 2, 0).cpu().numpy()

    # If values are normalized or in [0,1], this will still display fine.
    # If your images look dark/strange, you might need to rescale.
    plt.subplot(2, 3, i + 1)
    plt.imshow(img_np.squeeze(), cmap="gray" if img_np.shape[-1] == 1 else None)
    plt.title(f"Age: {labels_batch[i].item():.1f}")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
class AgeRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(AgeRegressionModel, self).__init__()

        # 4 fully connected (linear) layers
        self.fc1 = nn.Linear(input_dim, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 64)
        self.fc4 = nn.Linear(64, 1)   # Output layer (age prediction)

        self.relu = nn.ReLU()

    def forward(self, x):
        # Flatten image tensor (B, C, H, W) -> (B, C*H*W)
        x = x.view(x.size(0), -1)

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)   # No activation (regression)

        return x

In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()  # Set model to training mode
    running_loss = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)


In [ ]:
# Task 3: Write your validation loop here:
def validate(model, loader, criterion, device):
    model.eval()  # Set model to evaluation mode
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

    return running_loss / len(loader)


In [ ]:
# Task 4: Define device, model, loss, optimizer:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Get input dimension from one batch
sample_images, _ = next(iter(train_loader))
input_dim = sample_images.view(sample_images.size(0), -1).shape[1]

# Model
model = AgeRegressionModel(input_dim).to(device)

# Loss function (MAE for regression)
criterion = nn.L1Loss()

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
# Task 5: Start training for 20 epochs:
num_epochs = 20

train_losses = []
val_losses = []

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


In [ ]:
# Plot training and validation loss curves
plt.figure(figsize=(8,5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (MAE)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:



# Get one batch from the test loader
images, labels = next(iter(test_loader))

# Move images to device for prediction
images_device = images.to(device)

# Predict (no gradients needed)
model.eval()
with torch.no_grad():
    preds = model(images_device).cpu().numpy().squeeze()   # shape: (batch,)

# Convert labels to numpy
labels_np = labels.numpy().squeeze()

# Plot a few examples
num_show = 6
plt.figure(figsize=(12, 6))

for i in range(num_show):
    img = images[i]  # (C, H, W)

    # Convert from (C, H, W) -> (H, W, C) for matplotlib
    img_np = img.permute(1, 2, 0).numpy()

    # If image is normalized/looks dark, rescale to [0,1] for display
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)

    plt.subplot(2, 3, i + 1)
    plt.imshow(img_np.squeeze(), cmap="gray" if img_np.shape[-1] == 1 else None)
    plt.title(f"Pred: {preds[i]:.1f} | Actual: {labels_np[i]:.1f}")
    plt.axis("off")

plt.tight_layout()
plt.show()